#Upload and Read Excel file

In [1]:
from google.colab import files
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_rel

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print("PRS Excel file loaded successfully.")
display(df.head())
print(df.columns)

Saving PRS.xlsx to PRS.xlsx
PRS Excel file loaded successfully.


,ParticipantID,Condition,BA1,BA2,BA3,FAS4,FAS5,FAS6,COH7,COH8,COH9,SCO10,SCO11
0,P04,Smell,3,5,4,6,4,6,6,5,6,4,3
1,P04,NoSmell,5,5,5,5,6,6,6,5,6,4,2
2,P05,Smell,3,0,4,3,4,4,3,4,4,3,2
3,P05,NoSmell,4,0,3,3,4,3,3,4,4,3,2
4,P06,Smell,4,5,5,6,6,6,6,6,6,5,4


Index(['ParticipantID', 'Condition', 'BA1', 'BA2', 'BA3', 'FAS4', 'FAS5',
       'FAS6', 'COH7', 'COH8', 'COH9', 'SCO10', 'SCO11'],
      dtype='object')


#Check required columns

In [2]:
required_columns = [
    "ParticipantID",
    "Condition",
    "BA1", "BA2", "BA3",
    "FAS4", "FAS5", "FAS6",
    "COH7", "COH8", "COH9",
    "SCO10", "SCO11"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:")
    for col in missing_columns:
        print("-", col)
else:
    print("All required PRS columns are present.")

df["Condition"] = df["Condition"].astype(str).str.strip()

condition_mapping = {
    "smell": "Smell",
    "Smell": "Smell",
    "SMELL": "Smell",
    "no_smell": "NoSmell",
    "NoSmell": "NoSmell",
    "No-Smell": "NoSmell",
    "no-smell": "NoSmell",
    "No smell": "NoSmell",
    "no smell": "NoSmell",
    "control": "NoSmell",
    "Control": "NoSmell"
}

df["Condition"] = df["Condition"].replace(condition_mapping)

print(df["Condition"].value_counts())

All required PRS columns are present.
Condition
Smell      17
NoSmell    17
Name: count, dtype: int64


#Calculate PRS primary outcomes

In [3]:
# Calculate PRS dimension means
df["BeingAway"] = df[["BA1", "BA2", "BA3"]].mean(axis=1)

df["Fascination"] = df[["FAS4", "FAS5", "FAS6"]].mean(axis=1)

df["Coherence"] = df[["COH7", "COH8", "COH9"]].mean(axis=1)

df["Scope"] = df[["SCO10", "SCO11"]].mean(axis=1)

df["TotalPRS"] = df[
    [
        "BA1", "BA2", "BA3",
        "FAS4", "FAS5", "FAS6",
        "COH7", "COH8", "COH9",
        "SCO10", "SCO11"
    ]
].mean(axis=1)

# Keep only participant-level PRS scores
prs_scores = df[
    [
        "ParticipantID",
        "Condition",
        "TotalPRS",
        "BeingAway",
        "Fascination",
        "Coherence",
        "Scope"
    ]
].copy()

# Convert to wide format: one row per participant
wide_df = prs_scores.pivot(
    index="ParticipantID",
    columns="Condition",
    values=[
        "TotalPRS",
        "BeingAway",
        "Fascination",
        "Coherence",
        "Scope"
    ]
)

# Flatten column names
wide_df.columns = [
    f"{score}_{condition}"
    for score, condition in wide_df.columns
]

wide_df = wide_df.reset_index()

# Reorder columns
wide_df = wide_df[
    [
        "ParticipantID",
        "TotalPRS_NoSmell",
        "TotalPRS_Smell",
        "BeingAway_NoSmell",
        "BeingAway_Smell",
        "Fascination_NoSmell",
        "Fascination_Smell",
        "Coherence_NoSmell",
        "Coherence_Smell",
        "Scope_NoSmell",
        "Scope_Smell"
    ]
]

wide_df = wide_df.sort_values("ParticipantID")

print("Final participant-level PRS primary outcomes:")
display(wide_df)

primary_outcomes_file = "PRS_PrimaryOutcomes.xlsx"

wide_df.to_excel(
    primary_outcomes_file,
    index=False
)

print(f"{primary_outcomes_file} saved.")

Final participant-level PRS primary outcomes:


,ParticipantID,TotalPRS_NoSmell,TotalPRS_Smell,BeingAway_NoSmell,BeingAway_Smell,Fascination_NoSmell,Fascination_Smell,Coherence_NoSmell,Coherence_Smell,Scope_NoSmell,Scope_Smell
0,P04,5.000000,4.727273,5.000000,4.000000,5.666667,5.333333,5.666667,5.666667,3.0,3.5
1,P05,3.000000,3.090909,2.333333,2.333333,3.333333,3.666667,3.666667,3.666667,2.5,2.5
2,P06,4.090909,5.363636,3.666667,4.666667,4.333333,6.000000,5.000000,6.000000,3.0,4.5
3,P07,3.090909,2.909091,4.000000,3.000000,1.666667,2.000000,5.000000,4.000000,1.0,2.5
4,P08,3.545455,4.181818,4.666667,5.000000,3.666667,3.666667,3.000000,4.000000,2.5,4.0
5,P09,3.636364,5.181818,5.000000,5.666667,3.000000,4.666667,2.666667,4.666667,4.0,6.0
6,P10,4.363636,5.000000,4.333333,6.000000,5.000000,5.333333,4.666667,4.666667,3.0,3.5
7,P11,3.909091,4.818182,3.666667,5.000000,3.666667,5.000000,4.666667,4.333333,3.5,5.0
8,P12,2.727273,4.000000,1.333333,1.666667,4.666667,5.000000,2.666667,5.333333,2.0,4.0
9,P13,2.363636,2.727273,0.000000,0.000000,2.333333,3.666667,4.000000,5.333333,3.5,1.5


PRS_PrimaryOutcomes.xlsx saved.


#Descriptive statistics

In [4]:
descriptive_results = []

for measure_name, no_smell_col, smell_col in [
    ("Total PRS", "TotalPRS_NoSmell", "TotalPRS_Smell"),
    ("Being Away", "BeingAway_NoSmell", "BeingAway_Smell"),
    ("Fascination", "Fascination_NoSmell", "Fascination_Smell"),
    ("Coherence", "Coherence_NoSmell", "Coherence_Smell"),
    ("Scope", "Scope_NoSmell", "Scope_Smell")
]:

    descriptive_results.append({
        "PRS_Measure": measure_name,
        "Condition": "No-Smell",
        "Mean": wide_df[no_smell_col].mean(),
        "SD": wide_df[no_smell_col].std(ddof=1)
    })

    descriptive_results.append({
        "PRS_Measure": measure_name,
        "Condition": "Smell",
        "Mean": wide_df[smell_col].mean(),
        "SD": wide_df[smell_col].std(ddof=1)
    })

descriptive_df = pd.DataFrame(descriptive_results)

display(descriptive_df)

descriptive_file = "PRS_DescriptiveStatistics.xlsx" #Table 5.5

descriptive_df.to_excel(
    descriptive_file,
    index=False
)

print(f"{descriptive_file} saved.")

,PRS_Measure,Condition,Mean,SD
0,Total PRS,No-Smell,3.802139,0.830858
1,Total PRS,Smell,4.326203,0.852838
2,Being Away,No-Smell,3.607843,1.439692
3,Being Away,Smell,4.078431,1.605241
4,Fascination,No-Smell,3.882353,1.285286
5,Fascination,Smell,4.431373,1.116572
6,Coherence,No-Smell,4.529412,1.041225
7,Coherence,Smell,4.980392,0.691924
8,Scope,No-Smell,2.882353,1.256600
9,Scope,Smell,3.558824,1.367963


PRS_DescriptiveStatistics.xlsx saved.


#Shapiro–Wilk normality test

In [5]:
normality_results = []

for measure_name, no_smell_col, smell_col in [
    ("Total PRS", "TotalPRS_NoSmell", "TotalPRS_Smell"),
    ("Being Away", "BeingAway_NoSmell", "BeingAway_Smell"),
    ("Fascination", "Fascination_NoSmell", "Fascination_Smell"),
    ("Coherence", "Coherence_NoSmell", "Coherence_Smell"),
    ("Scope", "Scope_NoSmell", "Scope_Smell")
]:

    difference_scores = (
        wide_df[smell_col] - wide_df[no_smell_col]
    ).dropna()

    if len(difference_scores) >= 3:
        stat, p_value = shapiro(difference_scores)

        normality_results.append({
            "PRS_Measure": measure_name,
            "Shapiro_W": stat,
            "p_value": p_value,
            "Normality_Assumption": "Met" if p_value >= 0.05 else "Not met"
        })

    else:
        normality_results.append({
            "PRS_Measure": measure_name,
            "Shapiro_W": np.nan,
            "p_value": np.nan,
            "Normality_Assumption": "Not enough data"
        })

normality_df = pd.DataFrame(normality_results)

display(normality_df)

normality_file = "PRS_NormalityResults.xlsx" #Table F.3

normality_df.to_excel(
    normality_file,
    index=False
)

print(f"{normality_file} saved.")

,PRS_Measure,Shapiro_W,p_value,Normality_Assumption
0,Total PRS,0.955656,0.551794,Met
1,Being Away,0.924728,0.177489,Met
2,Fascination,0.931019,0.226348,Met
3,Coherence,0.903866,0.078856,Met
4,Scope,0.918209,0.137725,Met


PRS_NormalityResults.xlsx saved.


#Paired-samples t-test

In [6]:
ttest_results = []

for measure_name, no_smell_col, smell_col in [
    ("Total PRS", "TotalPRS_NoSmell", "TotalPRS_Smell"),
    ("Being Away", "BeingAway_NoSmell", "BeingAway_Smell"),
    ("Fascination", "Fascination_NoSmell", "Fascination_Smell"),
    ("Coherence", "Coherence_NoSmell", "Coherence_Smell"),
    ("Scope", "Scope_NoSmell", "Scope_Smell")
]:

    paired_data = wide_df[
        ["ParticipantID", no_smell_col, smell_col]
    ].dropna()

    no_smell_values = paired_data[no_smell_col]
    smell_values = paired_data[smell_col]

    difference_scores = smell_values - no_smell_values

    if len(paired_data) >= 2:
        t_stat, p_value = ttest_rel(
            smell_values,
            no_smell_values
        )

        difference_sd = difference_scores.std(ddof=1)

        cohens_d = (
            difference_scores.mean() / difference_sd
            if difference_sd != 0 else np.nan
        )

        ttest_results.append({
            "PRS_Measure": measure_name,
            "NoSmell_Mean": no_smell_values.mean(),
            "NoSmell_SD": no_smell_values.std(ddof=1),
            "Smell_Mean": smell_values.mean(),
            "Smell_SD": smell_values.std(ddof=1),
            "Mean_Difference_Smell_minus_NoSmell": difference_scores.mean(),
            "t_value": t_stat,
            "p_value": p_value,
            "Cohens_d": cohens_d
        })

    else:
        ttest_results.append({
            "PRS_Measure": measure_name,
            "NoSmell_Mean": np.nan,
            "NoSmell_SD": np.nan,
            "Smell_Mean": np.nan,
            "Smell_SD": np.nan,
            "Mean_Difference_Smell_minus_NoSmell": np.nan,
            "t_value": np.nan,
            "p_value": np.nan,
            "Cohens_d": np.nan
        })

ttest_results_df = pd.DataFrame(ttest_results)

display(ttest_results_df)

ttest_file = "PRS_PairedTTestResults.xlsx" #Table 5.6

ttest_results_df.to_excel(
    ttest_file,
    index=False
)

print(f"{ttest_file} saved.")

,PRS_Measure,NoSmell_Mean,NoSmell_SD,Smell_Mean,Smell_SD,Mean_Difference_Smell_minus_NoSmell,t_value,p_value,Cohens_d
0,Total PRS,3.802139,0.830858,4.326203,0.852838,0.524064,3.884426,0.001316,0.942112
1,Being Away,3.607843,1.439692,4.078431,1.605241,0.470588,2.260232,0.038104,0.548187
2,Fascination,3.882353,1.285286,4.431373,1.116572,0.549020,2.965912,0.009104,0.719339
3,Coherence,4.529412,1.041225,4.980392,0.691924,0.450980,2.037913,0.058444,0.494267
4,Scope,2.882353,1.256600,3.558824,1.367963,0.676471,2.669191,0.016798,0.647374


PRS_PairedTTestResults.xlsx saved.


#Download results

In [7]:
from google.colab import files

files.download("PRS_PrimaryOutcomes.xlsx")            # needed for master dataset
files.download("PRS_DescriptiveStatistics.xlsx")      #Table 5.6
files.download("PRS_NormalityResults.xlsx")           #Table F.3
files.download("PRS_PairedTTestResults.xlsx")         #Table 5.7

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>